# Module 3.3: The Full Transformer (End-To-End)

Welcome to the Grand Finale of Architecture Assembly! We have successfully built the math, the tokens, the positional wave functions, the powerful Encoder (Reader), and the autoregressive Decoder (Writer).

In this notebook, we wire the electrical box together and launch the complete 2017 `Seq2Seq` (Sequence-to-Sequence) Transformer.

## 1. The Grand Assembly (Stacking Layers)

### The Concept
A real Transformer is not just one Encoder block and one Decoder block. It is a massive stack of them. The original paper used 6 stacked layers. GPT-4 uses up to 96 layers! 

Below is the complete architectural flowchart of data moving through our network:

```mermaid
graph TD
    A[French Text: \"Bonjour\"] --> B(Token/Embedding + Position)
    B --> C[Encoder Block 1]
    C --> D[Encoder Block 2]
    D --> E[Encoder Block 3]
    
    F[English Prefix: \"Hello\"] --> G(Token/Embedding + Position)
    G --> H[Decoder Block 1]
    H --> I[Decoder Block 2]
    I --> J[Decoder Block 3]

    E -.->|Cross-Attention| H
    E -.->|Cross-Attention| I
    E -.->|Cross-Attention| J
```

### Why do we need it?
One layer isn't deep enough to reason. The first layer might discover syntax (nouns vs verbs). The second layer might discover sentiment (happy vs sad). By the 96th layer, the network has synthesized deep philosophical logic across the entire text!

In [1]:
import torch
import torch.nn as nn

# In PyTorch, we use nn.ModuleList() to create a stack of identical layers.
# It operates exactly like a Python list containing PyTorch modules!
dummy_layer = nn.Linear(128, 128)
stacked_layers = nn.ModuleList([dummy_layer for _ in range(6)])

print(f"We have stacked {len(stacked_layers)} layers!")

We have stacked 6 layers!


## 2. The Final Output (Linear + Softmax)

### The Concept
When the final Decoder Block finishes, it outputs a dense vector (e.g. `128` floating-point numbers). But that is useless to humans. We need it to pick an actual English word from our Vocabulary Dictionary (which might have `50,000` words)!

```mermaid
graph LR
    A[Decoder Output Vector\nDim: 128] -->|Linear Matrix| B(Logits Histogram\nDim: 50,000)
    B -->|Softmax Logic| C[Word: \"friend\"]
```

### Why do we need it? (Closing the Loop)
We use a massive `nn.Linear` matrix to mathematically stretch the `128` feature traits into `50,000` raw scores (Logits). The highest score represents the word the model is predicting! This perfectly closes the loop by connecting us straight back to the **Softmax Math** we learned in Notebook 02!

## 3. Building the End-To-End Transformer!

Let's write out the ultimate class. *(Note: For this notebook to run cleanly and instantly, we are mocking the interior blocks, but the architectural plumbing is 100% identical to the real code).* 

In [2]:
# --- Mocks from previous notebooks ---
class MockEncoderBlock(nn.Module):
    def forward(self, x): return x # Pretend this runs Attention & FFN

class MockDecoderBlock(nn.Module):
    def forward(self, x, enc_out, mask): return x # Pretend this runs Masked & Cross Attention
# --------------------------------------

class Transformer(nn.Module):
    def __init__(self, 
                 src_vocab_size: int,
                 tgt_vocab_size: int,
                 d_model: int = 256, 
                 num_layers: int = 6):
        super().__init__()
        
        # 1. Embeddings & Positions
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        # Note: We skip positional encoding logic here for simplicity, but it would be added to the embeddings.
        
        # 2. The Stacks
        self.encoder_stack = nn.ModuleList([MockEncoderBlock() for _ in range(num_layers)])
        self.decoder_stack = nn.ModuleList([MockDecoderBlock() for _ in range(num_layers)])
        
        # 3. The Final Vocabulary Projection
        self.final_linear = nn.Linear(d_model, tgt_vocab_size)

    def forward(self, src, tgt, tgt_mask):
        # Step A: Encoder reads the source text
        enc_x = self.src_embedding(src)
        for encoder in self.encoder_stack:
            enc_x = encoder(enc_x)
            
        # Step B: Decoder generates the target text (Masked and communicating with Encoder)
        dec_x = self.tgt_embedding(tgt)
        for decoder in self.decoder_stack:
            dec_x = decoder(dec_x, enc_out=enc_x, mask=tgt_mask)
            
        # Step C: Final Projection to Logits
        logits = self.final_linear(dec_x)
        
        return logits

# --- TEST TIME ---
# Pretend we have a dataset with 10k French words and 15k English words.
model = Transformer(src_vocab_size=10000, tgt_vocab_size=15000, d_model=256, num_layers=4)

# Send in 1 sentence, 10 French words
french_tokens = torch.randint(0, 10000, (1, 10))  
# Send in the English generation prefix (3 words so far)
english_prefix = torch.randint(0, 15000, (1, 3)) 
mask = torch.tril(torch.ones(3, 3))

final_prediction_logits = model(french_tokens, english_prefix, mask)
print(f"Input shape (English Prompt): {english_prefix.shape}")
print(f"Output Logits shape: {final_prediction_logits.shape} -> (Batch, Target_Seq, Vocab_Size!)")

Input shape (English Prompt): torch.Size([1, 3])
Output Logits shape: torch.Size([1, 3, 15000]) -> (Batch, Target_Seq, Vocab_Size!)


## 4. The Modern Split: Encoder-Only vs Decoder-Only

The architecture we just built is massive. It handles Sequence-to-Sequence (like translation or summarizing). But modern AI has realized we can just physically split the architecture in half to specialize in different tasks!

```mermaid
graph TD
    A[Sequence-to-Sequence\nThe Original 2017 Transformer] 
    A -->|Delete Decoder| B[Encoder-Only Architecture]
    A -->|Delete Encoder| C[Decoder-Only Architecture]
    
    B --> D{Google BERT}
    C --> E{ChatGPT / Llama 3}
    
    D -.->|Use Cases| F(Sentence Classification\nUnderstanding text\nSearching databases)
    E -.->|Use Cases| G(Talking to Humans\nGenerating Python Code\nWriting Essays)
```

### Why do we need it?
1. **Encoder-only (BERT)**: If you just need to classify a spam email, you don't need to generate text. You just need a model that reads the *entire* email at once and outputs a `True` or `False`. Encoder-only is perfectly optimized for this.
2. **Decoder-Only (GPT)**: If you want to chat with a bot, it doesn't need to translate anything. It just needs to aggressively read history and predict the very next word, over and over again. Decoder-Only models are leaner, faster, and scale incredibly well on modern GPUs.